<a href="https://colab.research.google.com/github/SergeiVKalinin/MSE_Fall_2026/blob/main/Module%202/Homework_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework: How Does the Data Change a Classifier?

In this homework, you will study how classification quality depends not only on the machine-learning algorithm, but also on the amount and structure of the data.

We will use MNIST handwritten digits as the primary classification problem. The target label is the digit identity, 0 through 9.

The homework has three subtasks:

1. **Learning curves on ordinary MNIST.**
2. **Adding irrelevant random information.**
3. **Adding a second visual object that is correlated, deterministically remapped, or unrelated to the MNIST label.**

You will compare three classifiers on the baseline problem:

- multinomial logistic regression;
- a multilayer perceptron;
- a support-vector machine.

The central question is:

> Does better classification performance mean that the model learned the intended digit information, or can performance improve because the dataset contains an easier shortcut?

Run the setup cell first. Do not change the random seed unless explicitly instructed.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tensorflow.keras.datasets import mnist
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

SEED = 17
rng = np.random.default_rng(SEED)

(X_train_img, y_train), (X_test_img, y_test) = mnist.load_data()

X_train_img = X_train_img.astype(np.float32) / 255.0
X_test_img = X_test_img.astype(np.float32) / 255.0

X_train = X_train_img.reshape(len(X_train_img), -1)
X_test = X_test_img.reshape(len(X_test_img), -1)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

def subset_xy(X, y, n, seed=SEED):
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(y), size=n, replace=False)
    return X[idx], y[idx]

def make_logistic():
    return LogisticRegression(max_iter=300, solver="lbfgs", random_state=SEED)

def make_mlp():
    return MLPClassifier(
        hidden_layer_sizes=(64,),
        activation="relu",
        max_iter=40,
        early_stopping=True,
        validation_fraction=0.1,
        random_state=SEED
    )

def make_svm():
    return SVC(kernel="rbf", C=5.0, gamma="scale")

models = {
    "Logistic regression": make_logistic,
    "MLP": make_mlp,
    "RBF SVM": make_svm,
}

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax, i in zip(axes.ravel(), range(10)):
    ax.imshow(X_train_img[i], cmap="gray")
    ax.set_title(f"label={y_train[i]}")
    ax.axis("off")
plt.tight_layout()
plt.show()


## Subtask 1 — Learning curves on ordinary MNIST

Use training-set sizes

$$N\in\{10, 30, 100, 500,1000,3000,10000, 30000\}$$

For each value of $N$:

1. draw a training subset;
2. train logistic regression, the MLP, and the RBF SVM;
3. calculate training accuracy;
4. calculate test accuracy on the fixed MNIST test set.

Create learning-curve plots showing training and test accuracy as functions of $N$.

### Questions

1. Which classifier performs best in the low-data regime?
2. Which classifier benefits most from increasing the amount of training data?
3. Is a large gap between training and test accuracy evidence of underfitting or overfitting?
4. At what point does adding more training data produce diminishing returns?
5. Does the ranking of classifiers depend on training-set size?


In [ ]:
train_sizes = [10, 30, 100, 500, 1000, 3000, 10000, 30000]

results_baseline = []

# TODO:
# Train all three classifiers for each training size.
# Record train and test accuracy.
# Plot learning curves.


## Subtask 2 — Append irrelevant random information

For every MNIST image, generate an independent $28\times28$ random-noise image and concatenate it to the right of the digit:

$$X_i^{\mathrm{noise}}=[D_i\mid R_i]$$

The random block contains no information about the class:

$$I(Y;R)=0$$

Use uniform random pixels between 0 and 1.

Use the **MLP** for the main comparison.

For each training size

$$N\in\{10, 30, 100, 500,1000,3000,10000, 30000\}$$

train on:

1. ordinary MNIST;
2. MNIST concatenated with independent random noise.

### Questions

1. Should the Bayes-optimal classifier improve after adding independent noise?
2. Does the finite-data MLP behave identically anyway?
3. Is the effect larger at small or large $N$?
4. Why can irrelevant dimensions hurt finite-data learning?
5. How is appending irrelevant information different from adding noise directly to the digit pixels?


In [ ]:
def append_random_noise(X_img, seed=0):
    # TODO: concatenate random 28x28 noise to the right
    pass

results_noise = []

# TODO:
# Compare MNIST and MNIST + random-noise block for the four N values.


## Subtask 3 — Chinese-character shortcuts

Append a rendered Chinese numeral to the right of each MNIST digit.

Use

$$0\rightarrow 零,\;1\rightarrow 一,\;2\rightarrow 二,\ldots,9\rightarrow 九$$

to make a $28\times56$ combined image.

Study three constructions.

### Case A — Matching mapping

The Chinese numeral represents the same numerical value as the MNIST digit.

### Case B — Wrong but deterministic mapping

Use one fixed permutation of the ten Chinese numerals. The mapping is semantically wrong but perfectly reproducible.

Therefore, the Chinese character is still a perfect predictor of the class.

### Case C — Random mapping

Choose the Chinese numeral independently at random for every sample.

Now the Chinese character contains no class information.

### Part 3A — In-distribution learning curves

Using the MLP, train all three constructions for

$$N\in\{10, 30, 100, 500,1000,3000,10000, 30000\}$$

and evaluate each on a test set constructed with the same mapping rule.

### Part 3B — Counterfactual shortcut tests

For models trained using matching and deterministic mappings, test on:

1. the same mapping;
2. randomized Chinese characters;
3. a new deterministic permutation;
4. MNIST digit only;
5. Chinese character only.

### Questions

1. Should matching and wrong-but-deterministic mappings have fundamentally different predictive difficulty?
2. Why is semantic correctness irrelevant to a purely predictive classifier?
3. What happens when the Chinese-character correlation is removed?
4. What happens when it is replaced with a new deterministic mapping?
5. Can excellent ordinary test accuracy coexist with learning the wrong feature?
6. Which signal did the model actually rely on?


In [ ]:
import os
import urllib.request
from PIL import Image, ImageDraw, ImageFont

FONT_PATH = "/tmp/NotoSansCJKsc-Regular.otf"

if not os.path.exists(FONT_PATH):
    font_url = (
        "https://raw.githubusercontent.com/notofonts/noto-cjk/"
        "main/Sans/OTF/SimplifiedChinese/NotoSansCJKsc-Regular.otf"
    )
    urllib.request.urlretrieve(font_url, FONT_PATH)

CHINESE_NUMERALS = ["零","一","二","三","四","五","六","七","八","九"]

def make_chinese_templates(size=28):
    font = ImageFont.truetype(FONT_PATH, 24)
    templates = []
    for char in CHINESE_NUMERALS:
        img = Image.new("L", (size, size), color=0)
        draw = ImageDraw.Draw(img)
        bbox = draw.textbbox((0, 0), char, font=font)
        w = bbox[2] - bbox[0]
        h = bbox[3] - bbox[1]
        x0 = (size - w) / 2 - bbox[0]
        y0 = (size - h) / 2 - bbox[1]
        draw.text((x0, y0), char, fill=255, font=font)
        templates.append(np.asarray(img, dtype=np.float32) / 255.0)
    return np.stack(templates)

CHINESE_TEMPLATES = make_chinese_templates()

WRONG_PERM = np.array([3, 7, 1, 8, 0, 9, 5, 2, 6, 4])
OOD_PERM = np.array([6, 2, 9, 0, 8, 4, 1, 7, 3, 5])

def append_chinese(X_img, y, mode="matching", seed=0, permutation=None):
    rng = np.random.default_rng(seed)

    if mode == "matching":
        chinese_id = y.copy()
    elif mode == "deterministic":
        if permutation is None:
            permutation = WRONG_PERM
        chinese_id = permutation[y]
    elif mode == "random":
        chinese_id = rng.integers(0, 10, size=len(y))
    else:
        raise ValueError("mode must be matching, deterministic, or random")

    chars = CHINESE_TEMPLATES[chinese_id]
    return np.concatenate([X_img, chars], axis=2)

def flatten_images(X_img):
    return X_img.reshape(len(X_img), -1)

def left_only(X_concat):
    X = X_concat.copy()
    X[:, :, 28:] = 0
    return X

def right_only(X_concat):
    X = X_concat.copy()
    X[:, :, :28] = 0
    return X

fig, axes = plt.subplots(2, 5, figsize=(8, 4))
for k, ax in enumerate(axes.ravel()):
    ax.imshow(CHINESE_TEMPLATES[k], cmap="gray")
    ax.set_title(f"{k}: {CHINESE_NUMERALS[k]}")
    ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
train_sizes = [10, 30, 100, 500, 1000, 3000, 10000, 30000]

results_chinese = []

# TODO Part 3A:
# Train MLPs using matching, deterministic, and random Chinese mappings.
# Plot test accuracy versus N.

# TODO Part 3B:
# For a selected N, evaluate shortcut models under:
# same mapping, randomized characters, new permutation,
# MNIST-only input, Chinese-only input.


## Final synthesis

Create one summary table comparing:

- ordinary MNIST;
- MNIST + random noise;
- MNIST + matching Chinese character;
- MNIST + deterministic wrong Chinese character;
- MNIST + random Chinese character.

Discuss the difference between:

- more samples;
- more input dimensions;
- more relevant information;
- irrelevant information;
- predictive but spurious information;
- in-distribution accuracy;
- robustness under distribution shift.

The final question is:

> What does a high test accuracy actually tell you about what the classifier learned?
